In [39]:
import cv2
import numpy as np

def overlay_heatmap(image, saliency, alpha=0.5):

    # image = (image - image.min()) / (image.max() - image.min() + 1e-8)
    # saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

    image_uint8 = np.uint8(255 * image)
    heatmap_uint8 = np.uint8(255 * saliency)

    heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(
        heatmap,
        alpha,
        np.stack([image_uint8]*3, axis=-1),
        1-alpha,
        0
    )

    return overlay

In [40]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/home/jovyan/work/MST")
sys.path.append(str(PROJECT_ROOT))

from mst.data.datasets.dataset_3d_odelia import ODELIA_Dataset3D

ds = ODELIA_Dataset3D(
    path_root="/home/jovyan/work/MST/mst/data/datasets/ODELIA_datasets",
    split="test"
)


In [41]:
import json

## Load the JSON file containing the correct UIDs for each class
with open("/home/jovyan/work/MST/scripts/Jupyter_notebook/Confusion Matrix/v3_correct_full.json", "r") as f:
    correct_data = json.load(f)

with open("/home/jovyan/work/MST/scripts/Jupyter_notebook/Confusion Matrix/v3_incorrect_full.json", "r") as f:
    incorrect_data = json.load(f)

saliency_set = {"GradCAM": "gradcam", "Raw Attention": "last_layer", "Slice Weighted Rollout": "slice_weighted_rollout"}



In [42]:
# data = {'0': ['ODELIA_TRICKS_0091_1_right']}

In [43]:
# def build_uid_index(dataset):
#     return {dataset[i]["uid"]: i for i in range(len(dataset))}

# uid_to_index = build_uid_index(ds)

In [44]:
import pickle
# with open("uid_to_index.pkl", "wb") as f:
#     pickle.dump(uid_to_index, f)

In [45]:
with open("uid_to_index.pkl", "rb") as f:
    uid_to_index = pickle.load(f)

In [49]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import gc

def files_exist(class_save_root, pred_class_name, file_name, formats):
    base_path = os.path.join(class_save_root, str(pred_class_name))
    for fmt in formats:
        fmt_folder = os.path.join(base_path, fmt[1:])
        save_path = os.path.join(fmt_folder, file_name + fmt)
        if not os.path.exists(save_path):
            return False
    return True

def save_saliency_visualizations(
    data,
    saliency_set,
    dataset,
    overlay_heatmap,
    save_root,
):

    os.makedirs(save_root, exist_ok=True)

    for gt_class, samples in data.items():
        class_save_root = os.path.join(save_root, gt_class)
        os.makedirs(class_save_root, exist_ok=True)


        for sample in samples:
            uid = sample["UID"]
            gt_class_name = sample["GT"]
            pred_class_name = sample["NN"]

            input = None
            idx = uid_to_index.get(uid)

            if idx is None:
                print(f"[ERROR] UID not found in dataset: {uid}")
                continue
            
            input = dataset[idx]
            num_slices = input["source"].shape[1]  # e.g., 32 slices

            volume = input["source"].squeeze(0).cpu()  # from [1,32,224,224] to [32,224,224]
            volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
            label = input["target"] # ground truth class index
            
            
            for method_name, saliency_map in saliency_set.items():

                file_formats = [".png", ".jpg"]
            
                overlay_file = f"{uid}_pred_{pred_class_name}_GT_{label}_{method_name}_overlay"
                comparison_file = f"{uid}_pred_{pred_class_name}_GT_{label}_{method_name}_comparison"
            
                # Skip early if BOTH outputs already exist
                overlay_done = files_exist(class_save_root, pred_class_name, overlay_file, file_formats)
                comparison_done = files_exist(class_save_root, pred_class_name, comparison_file, file_formats)
            
                if overlay_done and comparison_done:
                    print(f"[SKIP] Already processed: {uid} | {method_name}")
                    continue

                # -----------------------------
                # LOAD SALIENCY
                # -----------------------------
                saliency_path = (
                    f"/home/jovyan/work/MST/results/DINOv3ViTB/saliency_results/"
                    f"{saliency_map}/class_{label}/pt/{uid}_importance.pt"
                )

                if not os.path.exists(saliency_path):
                    print(f"[WARNING] Missing saliency: {saliency_path}")
                    continue

                saliency = torch.load(saliency_path, map_location="cpu", weights_only=True).numpy()

                print(f"Processing: {uid} | GT {label} | Pred {pred_class_name} | {method_name}")

                # =========================================================
                # 1) OVERLAY GRID
                # =========================================================
                overlays = []
                for i in range(num_slices):
                    overlays.append(
                        overlay_heatmap(volume[i], saliency[i])
                    )

                fig, axes = plt.subplots(4, 8, figsize=(20, 10))
                for i, ax in enumerate(axes.flat):
                    ax.imshow(overlays[i])
                    ax.set_title(f"S{i+1}", fontsize=8)
                    ax.axis("off")

                #fig.suptitle(f"{uid} | GT {label} | Pred {pred_class_name} | {method_name}", fontsize=14)


                plt.tight_layout()
                for fmt in file_formats:
                    fmt_folder = os.path.join(class_save_root, str(pred_class_name), fmt[1:])  # "png", "jpg"
                    os.makedirs(fmt_folder, exist_ok=True)

                    save_path = os.path.join(
                        fmt_folder,
                        overlay_file + fmt
                    )

                    plt.savefig(save_path, dpi=150 if fmt == ".png" else 100)
                # plt.savefig(overlay_path + ".png", dpi=150)
                # plt.savefig(overlay_path + ".jpg", dpi= 100)
                plt.close(fig)

                # =========================================================
                # 2) COMPARISON (RAW + OVERLAY)
                # =========================================================
                comparisons = []
                for i in range(num_slices):

                    img = volume[i]
                    sal = saliency[i]

                    overlay = overlay_heatmap(img, sal)

                    img_rgb = np.stack([img]*3, axis=-1)
                    # img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() + 1e-8)
                    img_rgb = np.uint8(255 * img_rgb)

                    combined = np.concatenate([img_rgb, overlay], axis=1)
                    comparisons.append(combined)

                fig, axes = plt.subplots(8, 4, figsize=(10, 12))
                for i, ax in enumerate(axes.flat):
                    ax.imshow(comparisons[i])
                    ax.set_title(f"S{i+1}", fontsize=8)
                    ax.axis("off")

                # fig.suptitle(f"{uid} | GT {label} | Pred {pred_class_name} | {method_name}", fontsize=14)


                plt.tight_layout()
                for fmt in file_formats:
                    fmt_folder = os.path.join(class_save_root, str(pred_class_name), fmt[1:])  # "png", "jpg"
                    os.makedirs(fmt_folder, exist_ok=True)

                    save_path = os.path.join(
                        fmt_folder,
                        comparison_file + fmt
                    )

                    plt.savefig(save_path, dpi=150 if fmt == ".png" else 100)
                # plt.savefig(comparison_path/"" + ".png", dpi=150)
                # plt.savefig(comparison_path + ".jpg", dpi = 100)
                plt.close(fig)

                # -----------------------------
                # MEMORY CLEANUP
                # -----------------------------
                del overlays, comparisons, saliency
                torch.cuda.empty_cache()
                gc.collect()
    print ("------------------------------------------------")
    print ("Finished processing all samples.")
    print ("------------------------------------------------")

In [50]:
save_saliency_visualizations(
    data=correct_data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image"
)

[SKIP] Already processed: ODELIA_BRAID1_0246_1_left | GradCAM
[SKIP] Already processed: ODELIA_BRAID1_0246_1_left | Raw Attention
[SKIP] Already processed: ODELIA_BRAID1_0246_1_left | Slice Weighted Rollout
[SKIP] Already processed: ODELIA_TRICKS_0067_1_right | GradCAM
[SKIP] Already processed: ODELIA_TRICKS_0067_1_right | Raw Attention
[SKIP] Already processed: ODELIA_TRICKS_0067_1_right | Slice Weighted Rollout
[SKIP] Already processed: ODELIA_BRAID1_0187_1_left | GradCAM
[SKIP] Already processed: ODELIA_BRAID1_0187_1_left | Raw Attention
[SKIP] Already processed: ODELIA_BRAID1_0187_1_left | Slice Weighted Rollout
[SKIP] Already processed: ODELIA_BRAID1_0777_1_left | GradCAM
[SKIP] Already processed: ODELIA_BRAID1_0777_1_left | Raw Attention
[SKIP] Already processed: ODELIA_BRAID1_0777_1_left | Slice Weighted Rollout
[SKIP] Already processed: ODELIA_TRICKS_0101_1_left | GradCAM
[SKIP] Already processed: ODELIA_TRICKS_0101_1_left | Raw Attention
[SKIP] Already processed: ODELIA_TRICKS

In [47]:
save_saliency_visualizations(
    data=incorrect_data,
    saliency_set=saliency_set,
    dataset=ds,
    overlay_heatmap=overlay_heatmap,
    save_root="/home/jovyan/work/MST/scripts/Jupyter_notebook/Overlay Image"
)

Processing: ODELIA_TRICKS_0091_1_left | GT 0 | Pred 2 | GradCAM
Processing: ODELIA_TRICKS_0091_1_left | GT 0 | Pred 2 | Raw Attention
Processing: ODELIA_TRICKS_0091_1_left | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: 08FEB48B_left | GT 0 | Pred 2 | GradCAM
Processing: 08FEB48B_left | GT 0 | Pred 2 | Raw Attention
Processing: 08FEB48B_left | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: 3E6C31E4_right | GT 0 | Pred 1 | GradCAM
Processing: 3E6C31E4_right | GT 0 | Pred 1 | Raw Attention
Processing: 3E6C31E4_right | GT 0 | Pred 1 | Slice Weighted Rollout
Processing: RUMC_069_left | GT 0 | Pred 2 | GradCAM
Processing: RUMC_069_left | GT 0 | Pred 2 | Raw Attention
Processing: RUMC_069_left | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: RUMC_035_right | GT 0 | Pred 2 | GradCAM
Processing: RUMC_035_right | GT 0 | Pred 2 | Raw Attention
Processing: RUMC_035_right | GT 0 | Pred 2 | Slice Weighted Rollout
Processing: UKA_11_left | GT 0 | Pred 1 | GradCAM
Processing: UKA_11_left